# 02 · Embeddings and Graph Construction

This notebook demonstrates:
1. How ESM-2 converts amino acid sequences into dense embedding vectors
2. How those embeddings are used to build protein graphs for GNN input

## 2.1 Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from src.embedder import ProteinEmbedder
from src.graph    import sequence_to_graph, build_sequential_edges, build_knn_edges

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2.2 Load ESM-2

We use the 35M parameter model for speed.  Replace with
`facebook/esm2_t33_650M_UR50D` (650M params) for best accuracy.

In [ ]:
embedder = ProteinEmbedder(
    model_name="facebook/esm2_t12_35M_UR50D",
    device=DEVICE,
)
print(f"Embedding dimension : {embedder.embed_dim}")

## 2.3 Single-sequence embedding

In [ ]:
tcr     = "CASSLAPGATNEKLFF"   # example TCR CDR3 (influenza-specific)
peptide = "GILGFVFTL"          # influenza M1 epitope

tcr_emb = embedder.embed_sequence(tcr)
pep_emb = embedder.embed_sequence(peptide)

print(f"TCR     length: {len(tcr):>3} aa → embedding shape: {tuple(tcr_emb.shape)}")
print(f"Peptide length: {len(peptide):>3} aa → embedding shape: {tuple(pep_emb.shape)}")

## 2.4 Visualise embedding similarity

In [ ]:
# Cosine similarity matrix between TCR residues
from torch.nn.functional import normalize, cosine_similarity

tcr_norm = normalize(tcr_emb, dim=1)
sim_matrix = (tcr_norm @ tcr_norm.T).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Similarity heatmap
sns.heatmap(
    sim_matrix,
    xticklabels=list(tcr),
    yticklabels=list(tcr),
    cmap="RdYlGn",
    vmin=-1, vmax=1,
    ax=axes[0],
)
axes[0].set_title(f"ESM-2 Residue Cosine Similarity\nTCR: {tcr}", fontweight="bold")

# PCA of TCR embeddings
pca    = PCA(n_components=2)
coords = pca.fit_transform(tcr_emb.numpy())
axes[1].scatter(coords[:, 0], coords[:, 1], s=120, alpha=0.9, c=range(len(tcr)),
                cmap="viridis")
for i, aa in enumerate(tcr):
    axes[1].annotate(aa, (coords[i, 0], coords[i, 1]),
                     fontsize=10, ha="center", va="bottom")
axes[1].set_xlabel("PC 1")
axes[1].set_ylabel("PC 2")
axes[1].set_title("PCA of TCR Residue Embeddings", fontweight="bold")

plt.tight_layout()
plt.show()

## 2.5 Graph construction

In [ ]:
g_tcr = sequence_to_graph(tcr, tcr_emb, k_neighbors=4)
g_pep = sequence_to_graph(peptide, pep_emb, k_neighbors=3)

print("TCR graph:")
print(f"  Nodes      : {g_tcr.num_nodes}")
print(f"  Edges      : {g_tcr.num_edges}")
print(f"  Node feats : {g_tcr.x.shape}")
print()
print("Peptide graph:")
print(f"  Nodes      : {g_pep.num_nodes}")
print(f"  Edges      : {g_pep.num_edges}")
print(f"  Node feats : {g_pep.x.shape}")

## 2.6 Visualise graph structure

In [ ]:
import networkx as nx
from torch_geometric.utils import to_networkx

def draw_protein_graph(graph, sequence, title):
    G = to_networkx(graph, to_undirected=True)
    pos = {i: (i, 0) for i in range(len(sequence))}  # linear layout

    fig, ax = plt.subplots(figsize=(max(8, len(sequence) * 0.8), 3))
    nx.draw_networkx_nodes(G, pos, node_size=600, node_color="#1f77b4",
                           alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos,
                            labels={i: aa for i, aa in enumerate(sequence)},
                            font_color="white", font_weight="bold", ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.3, ax=ax,
                           edge_color=["#ff7f0e" if abs(u-v)==1 else "#aec7e8"
                                       for u, v in G.edges()])
    ax.set_title(title, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

try:
    import networkx as nx
    draw_protein_graph(g_tcr, tcr, f"TCR Graph: {tcr}")
    draw_protein_graph(g_pep, peptide, f"Peptide Graph: {peptide}")
except ImportError:
    print("Install networkx for graph visualisation: pip install networkx")
    print(f"TCR graph has {g_tcr.num_edges} edges connecting {g_tcr.num_nodes} nodes.")

## 2.7 Key takeaways

- ESM-2 assigns *distinct* embeddings to each residue based on its sequence context — the same amino acid in a different position gets a different vector.
- The k-NN edges capture *functional similarity* without requiring a 3-D structure.
- Orange edges = sequential backbone; blue edges = k-NN (long-range).

Proceed to **notebook 03** to train the full GNN model ►